Does the candidate set keep getting richer past K = 15? **GPU T4 x2**, Internet on.

Attach `prepare-data-for-word-reranker` under Add Input → Your Work → Notebook Output.

In [1]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"
K = 30

In [2]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K_env

COMMIT = K_env.sync(REPO_URL, BRANCH)
K_env.gpu_info()
env = K_env.prepare(COMMIT)

Cloning into '/kaggle/working/candidate_reranker'...
From https://github.com/Splestule/candidate_reranker
 * branch            main       -> FETCH_HEAD


HEAD is now at b36958a k = 30 testing nb
torch 2.10.0+cu128  cuda=True
Tesla T4  compute capability 7.5
Turing/Pascal: no bf16, no FlashAttention 2 -- handled by wf_compat
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.4 MB/s eta 0:00:00


Cloning into '/kaggle/working/Whisfusion'...


commit      b36958a
base model  /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/mdm_safetensors/mdm-170M-100e18-rsl-0.01.safetensors
adapter     /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ckpt/whisfusion_stage2_decoder.pt
librispeech /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/data/LibriSpeech  ['dev-clean', 'test-clean', 'test-other']
hf cache    /kaggle/working/hf  prepared
ood         /kaggle/input/notebooks/eduardimon/prepare-data-for-word-reranker/ood  ['irish_english_male', 'midlands_english_female', 'northern_english_female']


K = 30 needs twice the activation memory of K = 15. If this cell reports out of memory, lower `K` to 24 and rerun from here.

In [3]:
rc = K_env.run(env, "selftest.py",
               "--base_model", env.base_model,
               "--adapter", env.adapter,
               "--librispeech", env.librispeech / "test-clean",
               "--n_utts", "3",
               "--n_candidates", K)
assert rc == 0, f"selftest exit code {rc}, see the output above"

[wf_compat] shims active: rotary_emb, dropout_layer_norm, flash_attn, xformers.ops.SwiGLU, FusedRMSNorm
SELFTEST
torch 2.10.0+cu128  cuda=True
GPU: Tesla T4  compute capability 7.5
     Turing/Pascal: no bf16, no FlashAttention 2 -> fp16 + SDPA
[1/5] compat shims OK
[wf] device=cuda (Tesla T4) dtype=torch.float16
[wf] base: 129 keys, 72 missing
[wf] adapter: 201 keys, 0 missing, 0 unexpected

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1833.87it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]
[wf] decoder 261.5M · Whisper encoder 88.2M
[2/5] checkpoint loaded with no missing keys
  1089-134686-0000  10.4s  WER  10.7 (oracle  10.7, 30/30 unique)
      REF he hoped there would be stew for dinner turnips and carrots and brui
      HYP he hoped there would be stew for dinner turnips and carrots and b br
  1089-134686-0001   3.3s  WER  37.5 (oracle  25.0, 15/30 unique)
      REF stuff it into you his belly counselled him
      HYP stuffed into you his

Roughly 4 hours for test-clean at K = 30. `--resume` picks up if the session dies.

In [4]:
K_env.run(env, "dump_candidates.py",
          "--source", "librispeech", "--path", env.librispeech / "test-clean",
          "--base_model", env.base_model, "--adapter", env.adapter,
          "--n_candidates", K,
          "--out", env.results / f"test-clean-k{K}-{COMMIT}.jsonl",
          "--tag", "test-clean", "--resume")

[wf_compat] shims active: rotary_emb, dropout_layer_norm, flash_attn, xformers.ops.SwiGLU, FusedRMSNorm
[wf] device=cuda (Tesla T4) dtype=torch.float16
[wf] base: 129 keys, 72 missing
[wf] adapter: 201 keys, 0 missing, 0 unexpected

Loading weights: 100%|██████████| 479/479 [00:00<00:00, 1860.65it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]
[wf] decoder 261.5M · Whisper encoder 88.2M
  25 utts · 0.7 min · 1.72 s/utt · RTF 0.200
  50 utts · 1.5 min · 1.78 s/utt · RTF 0.237
  75 utts · 2.3 min · 1.83 s/utt · RTF 0.222
  100 utts · 3.1 min · 1.84 s/utt · RTF 0.204
  125 utts · 3.9 min · 1.85 s/utt · RTF 0.216
  150 utts · 4.6 min · 1.86 s/utt · RTF 0.214
  175 utts · 5.4 min · 1.86 s/utt · RTF 0.217
  200 utts · 6.2 min · 1.87 s/utt · RTF 0.205
  225 utts · 7.0 min · 1.87 s/utt · RTF 0.208
  250 utts · 7.8 min · 1.87 s/utt · RTF 0.213
  275 utts · 8.6 min · 1.87 s/utt · RTF 0.212
  300 utts · 9.4 min · 1.88 s/utt · RTF 0.213
  325 utts · 10.2 min · 1.88 s/u

0

First look. The ladder is derived from this one dump, so every rung sees the same utterances.

In [5]:
DUMP = env.results / f"test-clean-k{K}-{COMMIT}.jsonl"

K_env.run(env, "analyze.py", DUMP, "--json", env.results / f"test-clean-k{K}-{COMMIT}.stats.json")

for k in [5, 10, 15, 20, 25, K]:
    print("\n" + "#" * 74 + f"\n# composition at k = {k}\n" + "#" * 74)
    K_env.run(env, "analyze_compose.py", DUMP,
              "--max_k", k, "--alpha", 0.5, "--eps_conf", 0.7, "--gamma", 1.0,
              "--n_boot", 1000,
              "--json", env.results / f"compose-k{k}-{COMMIT}.json")

ORACLE GAP  ·  /kaggle/working/results/test-clean-k30-b36958a.jsonl
utterances: 2620   K: 30   audio: 324.2 min
identical candidates after step 1: 100.0 % of utts

--------------------------------------------------------------------------
scorer                        corpus WER   mean-utt     hit %       gap
--------------------------------------------------------------------------
mean_conf (upstream)                8.58       7.88      65.4      2.81
min_conf                            9.95       8.80      59.9      3.73
median_conf                         9.90       9.19      53.6      4.12
mean_logprob                        8.61       7.84      66.0      2.77
neg_entropy                         8.68       7.90      65.4      2.83
len_norm_conf                       8.87       8.44      61.0      3.36
mbr_wer                             7.79       7.35      72.0      2.28
conf+0.5*mbr                        7.88       7.31      72.0      2.24
--------------------------------------